# DharmaOCR — Notebook 02: validação por k-fold e comparação com outros OCRs

Segundo notebook do POC. Objetivo: **validar os números do paper** (arXiv:2604.14314) com barras de erro via **k-fold** e **comparar o Dharma OCR com outros OCRs** na *mesma* figura de mérito.

## O que é o "k-fold" aqui (e por que é barato)

O benchmark é um conjunto de **avaliação** (496 docs, sem treino). Então k-fold aqui **não** é cross-validation de treino: é **reamostragem** do score do benchmark para estimar variância/estabilidade.

- Roda-se a inferência de cada OCR **uma única vez** sobre os 496 docs → score por documento, cacheado em disco.
- Particiona-se os documentos em **K folds**; para cada fold calcula-se o `benchmark_score = (mean(Lev) + mean(BLEU))/2` daquele fold.
- Reporta-se **média ± desvio** (e IC 95%) dos K folds por modelo.

Assim comparamos OCRs com **barras de erro** (não um ponto só) e checamos se o número do paper é reprodutível/estável. **Custo = ~496 chamadas por OCR, uma vez** — o k-fold só reamostra os scores já calculados.

## Fidelidade à implementação oficial

A métrica replica **exatamente** o `evaluation.ipynb` publicado junto ao dataset:

- **Levenshtein_Ratio** = `1 − distance / max(len)` (0.0 quando ambos vazios).
- **BLEU** = NLTK `sentence_bleu([gt.split()], [pred.split()], SmoothingFunction().method1)` — BLEU‑4 sobre tokens `split()`. **Não** é sacrebleu.
- **Ground‑truth** = coluna `assistant` (JSON) normalizada por `PLAIN_TEXT` (junta os valores do JSON com `\n`); a predição passa pela mesma normalização.
- **benchmark_score** = `(mean(Lev) + mean(BLEU)) / 2`.
- **text degeneration** = atingiu o limite de tokens (`finish_reason == "length"`) **e** os últimos 15 caracteres se repetem ≥4× na saída.

## Subconjuntos

O release **não traz rótulo de subconjunto** por linha (ESTER‑Pt/Legal/BRESSAY estão interleaved no `id`), então o k-fold é sobre os 496 como um todo. Para quebra por subconjunto seria preciso obter os rótulos com a Dharma.


## 0. Dependências e parâmetros

In [ ]:
# Colab: datasets + rapidfuzz sempre; vllm só se houver GPU (para GLM/Baidu self-host).
!pip install -q datasets rapidfuzz
!command -v nvidia-smi >/dev/null 2>&1 && pip install -q vllm || echo "(sem GPU — pulando vllm; GLM/Baidu precisam de GPU)"
# (Local Python 3.8: use os pins  "datasets<3.0" "huggingface_hub<0.24" "nltk<3.9".)
print("deps prontas.")

In [ ]:
import os, re, json, time, base64, glob, math
from dataclasses import dataclass
from typing import Callable, Optional, List, Dict, Any

import numpy as np
import pandas as pd

# ---- Parâmetros (edite aqui) --------------------------------------------------
DATASET_ID   = "Dharma-AI/DharmaOCR-Benchmark"
SPLIT        = "test"
GT_COLUMN    = "assistant"          # ground-truth JSON (como no evaluation.ipynb oficial)
PLAIN_TEXT   = True                 # normaliza JSON -> texto puro antes da métrica

SAMPLE_N     = 20                   # piloto (int). None = 496 (completo)
KFOLD_K      = 5                    # nº de folds
SEED         = 42                   # semente do embaralhamento dos folds
CACHE_DIR    = "benchmark_results"  # scores por-doc de cada modelo vão aqui

# Prompt de sistema idêntico ao evaluation.ipynb oficial (usado pelos leitores VLM/LLM)
DEFAULT_SYSTEM_PROMPT = (
    "You are an expert OCR system. Extract all text from the provided document image "
    "exactly as it appears, preserving the original layout and structure. "
    "Return only the extracted text with no additional commentary."
)
HTTP_TIMEOUT = int(os.environ.get("OCR_HTTP_TIMEOUT", "180"))
os.makedirs(CACHE_DIR, exist_ok=True)
print("parâmetros OK | K =", KFOLD_K, "| SAMPLE_N =", SAMPLE_N, "| cache =", CACHE_DIR)

## 1. Dataset

Carregamos o split `test`. Passamos `image_base64` para os leitores e `assistant` como ground‑truth.

In [ ]:
from datasets import load_dataset
ds = load_dataset(DATASET_ID, split=SPLIT)
print(ds, "\nn =", len(ds), "| colunas:", ds.column_names)

## 2. Figura de mérito (fiel ao `evaluation.ipynb` oficial)

In [ ]:
from rapidfuzz.distance import Levenshtein as _Lev            # distância de edição inteira (igual à python-Levenshtein)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
_SMOOTHER = SmoothingFunction().method1


def json_to_plain(x: Any) -> str:
    """Replica _json_to_plain do notebook oficial: se for JSON dict, junta os valores
    não-vazios com '\n'; caso contrário devolve como texto."""
    try:
        parsed = json.loads(str(x))
    except (json.JSONDecodeError, TypeError):
        return str(x)
    if isinstance(parsed, dict):
        return "\n".join(str(v) for v in parsed.values() if v is not None and str(v).strip())
    return str(x)


def levenshtein_ratio(a: str, b: str) -> float:
    a = "" if a is None else str(a); b = "" if b is None else str(b)
    m = max(len(a), len(b))
    return (1.0 - _Lev.distance(a, b) / m) if m > 0 else 0.0


def bleu(gt: str, pred: str) -> float:
    # ordem oficial: referência = ground-truth, hipótese = predição
    return float(sentence_bleu([str(gt).split()], str(pred).split(), smoothing_function=_SMOOTHER))


def score_pair(gt_raw: Any, pred_raw: Any, plain_text: bool = True) -> Dict[str, Any]:
    gt   = json_to_plain(gt_raw)   if plain_text else str(gt_raw)
    pred = json_to_plain(pred_raw) if plain_text else str(pred_raw)
    lev = levenshtein_ratio(gt, pred); bl = bleu(gt, pred)
    return {"levenshtein_ratio": lev, "bleu": bl, "score": (lev + bl) / 2.0}


# ---- text degeneration (definição oficial) -----------------------------------
def _tail_repeats(text: str, tail: int = 15, times: int = 4) -> bool:
    t = str(text)
    if len(t) < tail:
        return False
    return t.count(t[-tail:]) >= times

def is_degenerate(text: str, hit_token_limit: bool) -> bool:
    """Oficial: atingiu limite de tokens E os últimos 15 chars se repetem >= 4x."""
    return bool(hit_token_limit) and _tail_repeats(text)

In [ ]:
# Sanity check da métrica em linhas reais do benchmark
_ex = ds[0]
s_ok  = score_pair(_ex["assistant"], _ex["assistant_without_json"])   # pred ~ gt
s_bad = score_pair(_ex["assistant"], _ex["assistant_without_json"][:len(_ex["assistant_without_json"])//2])
print("pred≈gt :", {k: round(v,3) for k,v in s_ok.items()})
print("pred/2  :", {k: round(v,3) for k,v in s_bad.items()})
assert s_ok["score"] > s_bad["score"]
assert is_degenerate("abcabcabcabc"*5, hit_token_limit=True) is True
assert is_degenerate("texto normal sem repetição", hit_token_limit=True) is False
assert is_degenerate("abcabcabcabc"*5, hit_token_limit=False) is False   # precisa do limite de tokens
print("OK: métrica e detector de degeneração fiéis ao oficial.")

## 3. Leitores (adapters de OCR)

Interface única `OCRReader: image_base64 -> OCRResult`. Todos configuráveis por env — **nenhuma chave no código**.

Os três modelos são **open-weights (MIT)** e podem rodar via **vLLM self-hosted** (endpoint OpenAI-compatible), com o mesmo adapter `make_openai_ocr_reader` e o system prompt oficial — comparação uniforme, reproduzível e sem custo por chamada:

| Modelo | Licença | Acesso (padrão) | Config (env) |
|---|---|---|---|
| **Dharma-OCR full / lite** | open-weights (LITE público, FULL *gated*) | API hospedada `/v1/ocrs/` (nb 01); ou vLLM se `DHARMA_VLLM_URL` setado | `DHARMA_API_KEY` · ou `DHARMA_VLLM_URL` (+ `DHARMA_VLLM_*`) |
| **Baidu Unlimited-OCR** | MIT | vLLM self-host | `BAIDU_OCR_URL`, `BAIDU_OCR_KEY`, `BAIDU_OCR_MODEL` |
| **GLM-OCR** | MIT | vLLM self-host (`GLM_OCR_URL`); *fallback* Z.ai hospedado (`GLM_API_KEY`) | `GLM_OCR_URL` (+ `GLM_OCR_*`) · ou `GLM_API_KEY` |

> Subir um modelo aberto e apontar a env: `vllm serve zai-org/GLM-OCR --port 8002` → `GLM_OCR_URL=http://localhost:8002/v1` (idem `baidu/Unlimited-OCR` e `Dharma-AI/Dharma-OCR-LITE`). O adapter OpenAI-compatible manda a imagem como data URI base64 e lê `choices[0].message.content`.


In [ ]:
import requests

@dataclass
class OCRResult:
    text: str
    raw: Any = None
    hit_token_limit: bool = False
    latency_s: float = 0.0
    out_tokens: Optional[int] = None
    error: Optional[str] = None

OCRReader = Callable[[str], OCRResult]

def to_data_uri(image_base64: str, mime: str = "image/png") -> str:
    s = image_base64 or ""
    if s.startswith("data:"):
        return s
    if s.startswith("/9j/"):   mime = "image/jpeg"
    elif s.startswith("iVBOR"): mime = "image/png"
    elif s.startswith("UklGR"): mime = "image/webp"
    return f"data:{mime};base64,{s}"

# ---------- Dharma OCR (API HTTP /v1/ocrs/, do notebook 01) --------------------
DHARMA_API_URL = os.environ.get("DHARMA_API_URL", "https://ocr-api.com-us-east-2.dharma-ai.com/v1/ocrs/")
DHARMA_API_KEY = os.environ.get("DHARMA_API_KEY", "")

def dharma_read(image_base64: str, model: str = "full") -> OCRResult:
    if not DHARMA_API_KEY:
        return OCRResult(text="", error="DHARMA_API_KEY não configurada")
    payload = {"b64image": to_data_uri(image_base64), "model": model}
    headers = {"Authorization": f"Bearer {DHARMA_API_KEY}", "Content-Type": "application/json"}
    t0 = time.time()
    try:
        r = requests.post(DHARMA_API_URL, json=payload, headers=headers, timeout=HTTP_TIMEOUT)
        dt = time.time() - t0; r.raise_for_status(); d = r.json()
        meta = d.get("metadata") or {}
        return OCRResult(text=d.get("text", "") or "", raw=d,
                         hit_token_limit=bool(meta.get("truncated_pages")),
                         latency_s=dt, out_tokens=meta.get("num_output_tokens"))
    except Exception as e:
        return OCRResult(text="", latency_s=time.time() - t0, error=str(e))

In [ ]:
# ---------- Leitor genérico OpenAI-compatible (chat/completions com imagem) ----
# Serve modelos abertos self-hosted via vLLM: Baidu Unlimited-OCR, GLM-OCR (modo vLLM), etc.
def make_openai_ocr_reader(base_url: str, api_key: str, model: str,
                           system_prompt: str = DEFAULT_SYSTEM_PROMPT,
                           max_tokens: int = 8192) -> OCRReader:
    url = base_url.rstrip("/") + "/chat/completions"
    def _reader(image_base64: str) -> OCRResult:
        payload = {"model": model, "temperature": 0, "max_tokens": max_tokens,
                   "messages": [
                       {"role": "system", "content": system_prompt},
                       {"role": "user", "content": [
                           {"type": "image_url", "image_url": {"url": to_data_uri(image_base64)}}]},
                   ]}
        headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
        t0 = time.time()
        try:
            r = requests.post(url, json=payload, headers=headers, timeout=HTTP_TIMEOUT)
            dt = time.time() - t0; r.raise_for_status(); d = r.json()
            ch = (d.get("choices") or [{}])[0]
            text = ((ch.get("message") or {}).get("content")) or ""
            usage = d.get("usage") or {}
            return OCRResult(text=text, raw=d,
                             hit_token_limit=(ch.get("finish_reason") == "length"),
                             latency_s=dt, out_tokens=usage.get("completion_tokens"))
        except Exception as e:
            return OCRResult(text="", latency_s=time.time() - t0, error=str(e))
    return _reader


# ---------- GLM-OCR (Z.ai layout_parsing, hospedado) --------------------------
def _extract_glm_text(d: Any) -> str:
    """Extrai texto/markdown da resposta do layout_parsing de forma defensiva
    (o shape exato pode variar; pega a maior string sob chaves prováveis)."""
    if d is None:
        return ""
    if isinstance(d, str):
        return d
    keys = ("markdown", "md", "text", "content", "result", "data", "output", "results")
    best = ""
    def walk(o):
        nonlocal best
        if isinstance(o, str):
            if len(o) > len(best):
                best = o
        elif isinstance(o, dict):
            for k in keys:
                if k in o:
                    walk(o[k])
            for v in o.values():
                walk(v)
        elif isinstance(o, list):
            for v in o:
                walk(v)
    walk(d)
    return best

def make_glm_ocr_reader(api_key: str,
                        base_url: str = "https://api.z.ai/api/paas/v4",
                        model: str = "glm-ocr") -> OCRReader:
    url = base_url.rstrip("/") + "/layout_parsing"
    def _reader(image_base64: str) -> OCRResult:
        payload = {"model": model, "file": to_data_uri(image_base64)}   # 'file' aceita URL; base64 data URI best-effort
        headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
        t0 = time.time()
        try:
            r = requests.post(url, json=payload, headers=headers, timeout=HTTP_TIMEOUT)
            dt = time.time() - t0; r.raise_for_status(); d = r.json()
            return OCRResult(text=_extract_glm_text(d), raw=d, latency_s=dt)
        except Exception as e:
            return OCRResult(text="", latency_s=time.time() - t0, error=str(e))
    return _reader

In [ ]:
# ---------- Registro de modelos (só entra quem estiver configurado) -----------
def make_oracle_reader(dataset) -> OCRReader:
    """Leitor 'oráculo' (copia o ground-truth em texto puro) — valida o pipeline sem gastar API."""
    ref = {i: dataset[i]["assistant_without_json"] for i in range(len(dataset))}
    def _reader(image_base64: str, _p=[0]) -> OCRResult:
        i = _p[0]; _p[0] += 1
        return OCRResult(text=ref.get(i, ""))
    return _reader

# Os três modelos são open-weights (MIT) e rodam via vLLM self-hosted (endpoint OpenAI-compatible),
# usando o MESMO adapter make_openai_ocr_reader e o system prompt oficial.
#   vllm serve zai-org/GLM-OCR       --port 8002   -> GLM_OCR_URL=http://localhost:8002/v1
#   vllm serve baidu/Unlimited-OCR   --port 8001   -> BAIDU_OCR_URL=http://localhost:8001/v1
#   vllm serve Dharma-AI/Dharma-OCR-LITE --port 8003 -> DHARMA_VLLM_URL=http://localhost:8003/v1

# Baidu Unlimited-OCR (open-weights)
BAIDU_OCR_URL   = os.environ.get("BAIDU_OCR_URL", "")          # ex.: http://localhost:8001/v1
BAIDU_OCR_KEY   = os.environ.get("BAIDU_OCR_KEY", "EMPTY")
BAIDU_OCR_MODEL = os.environ.get("BAIDU_OCR_MODEL", "baidu/Unlimited-OCR")

# GLM-OCR (open-weights): vLLM self-host tem prioridade; fallback = Z.ai hospedado (pago)
GLM_OCR_URL     = os.environ.get("GLM_OCR_URL", "")            # ex.: http://localhost:8002/v1  (vLLM)
GLM_OCR_KEY     = os.environ.get("GLM_OCR_KEY", "EMPTY")
GLM_OCR_MODEL   = os.environ.get("GLM_OCR_MODEL", "zai-org/GLM-OCR")
GLM_API_KEY     = os.environ.get("GLM_API_KEY", "")            # só usado se GLM_OCR_URL vazio
GLM_BASE_URL    = os.environ.get("GLM_BASE_URL", "https://api.z.ai/api/paas/v4")
GLM_HOSTED_MODEL = os.environ.get("GLM_MODEL", "glm-ocr")

# Dharma (open-weights): opção de rodar via vLLM em vez da API hospedada /v1/ocrs/.
# Obs.: Dharma-AI/Dharma-OCR-LITE é público; Dharma-AI/Dharma-OCR-FULL é gated (peça acesso no HF).
DHARMA_VLLM_URL  = os.environ.get("DHARMA_VLLM_URL", "")       # ex.: http://localhost:8003/v1
DHARMA_VLLM_KEY  = os.environ.get("DHARMA_VLLM_KEY", "EMPTY")
DHARMA_VLLM_FULL = os.environ.get("DHARMA_VLLM_FULL_MODEL", "Dharma-AI/Dharma-OCR-FULL")
DHARMA_VLLM_LITE = os.environ.get("DHARMA_VLLM_LITE_MODEL", "Dharma-AI/Dharma-OCR-LITE")

MODELS: Dict[str, OCRReader] = {}

# Dharma: vLLM self-host tem prioridade se configurado; senão a API hospedada do nb 01
if DHARMA_VLLM_URL:
    MODELS["Dharma-OCR-full"] = make_openai_ocr_reader(DHARMA_VLLM_URL, DHARMA_VLLM_KEY, DHARMA_VLLM_FULL)
    MODELS["Dharma-OCR-lite"] = make_openai_ocr_reader(DHARMA_VLLM_URL, DHARMA_VLLM_KEY, DHARMA_VLLM_LITE)
elif DHARMA_API_KEY:
    MODELS["Dharma-OCR-full"] = lambda b: dharma_read(b, "full")
    MODELS["Dharma-OCR-lite"] = lambda b: dharma_read(b, "lite")

# Baidu Unlimited-OCR (vLLM)
if BAIDU_OCR_URL:
    MODELS["Unlimited-OCR"] = make_openai_ocr_reader(BAIDU_OCR_URL, BAIDU_OCR_KEY, BAIDU_OCR_MODEL)

# GLM-OCR: vLLM self-host (padrão) ou Z.ai hospedado (fallback)
if GLM_OCR_URL:
    MODELS["GLM-OCR"] = make_openai_ocr_reader(GLM_OCR_URL, GLM_OCR_KEY, GLM_OCR_MODEL)
elif GLM_API_KEY:
    MODELS["GLM-OCR"] = make_glm_ocr_reader(GLM_API_KEY, GLM_BASE_URL, GLM_HOSTED_MODEL)

print("Modelos configurados:", list(MODELS.keys()) or "(nenhum — configure as envs; usarei o oráculo no smoke test)")

### 3.1 (Colab/GPU) Subir modelos abertos via vLLM

No Colab com runtime **GPU (T4, 15 GB)**, suba **um modelo aberto por vez** via vLLM e rode a inferência dele — a T4 comporta um modelo pequeno/médio de cada vez. Fluxo por modelo:

```python
start_vllm("zai-org/GLM-OCR", 8002, "GLM_OCR_URL")                       # GLM-OCR 0.9B (cabe folgado)
MODELS["GLM-OCR"] = make_openai_ocr_reader(os.environ["GLM_OCR_URL"], "EMPTY", "zai-org/GLM-OCR")
run_inference("GLM-OCR", MODELS["GLM-OCR"], ds, n=40)                    # cacheia os scores
stop_vllm(8002)                                                          # libera a GPU

start_vllm("baidu/Unlimited-OCR", 8001, "BAIDU_OCR_URL", max_model_len=8192)   # ~3B, cabe na T4
MODELS["Unlimited-OCR"] = make_openai_ocr_reader(os.environ["BAIDU_OCR_URL"], "EMPTY", "baidu/Unlimited-OCR")
run_inference("Unlimited-OCR", MODELS["Unlimited-OCR"], ds, n=40)
stop_vllm(8001)
```

> **Dharma no Colab:** mantenha a **API hospedada** (`/v1/ocrs/`, só a chave) — os 3B/7B do Dharma são pesados para a T4. Se quiser o LITE (3B) via vLLM: `start_vllm("Dharma-AI/Dharma-OCR-LITE", 8003, "DHARMA_VLLM_URL")` e re-rode a célula do registro. O FULL (7B) é *gated* e aperta os 15 GB — prefira a API.


In [ ]:
# --- (Colab/GPU) Helpers para subir modelos abertos via vLLM ------------------
# T4 (15GB) comporta ~1 modelo de cada vez: start_vllm -> run_inference -> stop_vllm -> próximo.
import subprocess, time
import requests as _rq

_VLLM_PROCS: Dict[int, Any] = {}

def start_vllm(model: str, port: int, env_url_var: str,
               max_model_len: int = 8192, gpu_mem: float = 0.90,
               extra_args: Optional[list] = None, startup_timeout: int = 1800) -> "subprocess.Popen":
    """Sobe `vllm serve <model>` como subprocesso e aponta os.environ[env_url_var] para ele.

    Ex.: start_vllm("zai-org/GLM-OCR", 8002, "GLM_OCR_URL")
         MODELS["GLM-OCR"] = make_openai_ocr_reader(os.environ["GLM_OCR_URL"], "EMPTY", "zai-org/GLM-OCR")
         run_inference("GLM-OCR", MODELS["GLM-OCR"], ds, n=40)
         stop_vllm(8002)
    """
    cmd = ["vllm", "serve", model, "--host=127.0.0.1", f"--port={port}",
           "--trust-remote-code", "--dtype=auto",
           f"--gpu-memory-utilization={gpu_mem}", f"--max-model-len={max_model_len}"]
    if extra_args:
        cmd += list(extra_args)
    log_path = f"vllm_{port}.log"
    logf = open(log_path, "w")
    proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT)
    _VLLM_PROCS[port] = (proc, logf)
    health = f"http://127.0.0.1:{port}/health"
    deadline = time.time() + startup_timeout
    while time.time() < deadline:
        if proc.poll() is not None:
            logf.flush()
            raise RuntimeError(f"vLLM saiu antes de subir — veja {log_path}.")
        try:
            if _rq.get(health, timeout=2).status_code == 200:
                os.environ[env_url_var] = f"http://127.0.0.1:{port}/v1"
                print(f"vLLM pronto: {model} -> {os.environ[env_url_var]}")
                return proc
        except Exception:
            pass
        time.sleep(3)
    raise TimeoutError(f"vLLM não ficou pronto em {startup_timeout}s — veja {log_path}.")

def stop_vllm(port: int) -> None:
    item = _VLLM_PROCS.pop(port, None)
    if not item:
        print("nenhum vLLM na porta", port); return
    proc, logf = item
    proc.terminate()
    try:
        proc.wait(timeout=30)
    except Exception:
        proc.kill()
    logf.close()
    print("vLLM parado (porta", port, ")")

# Checagem rápida de GPU (informativo)
try:
    _smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                          capture_output=True, text=True, timeout=15)
    print("GPU:", _smi.stdout.strip() or "(nvidia-smi sem saída)")
except Exception:
    print("Sem GPU detectada (nvidia-smi ausente) — use as APIs hospedadas para GLM/Dharma.")

## 4. Inferência-uma-vez com cache

`run_inference` roda um leitor sobre N docs, calcula o score por documento e **salva em disco** (`benchmark_results/<modelo>.parquet`, com fallback CSV). Re-execuções reutilizam o cache — as APIs não são chamadas de novo. É aqui que o custo acontece (uma vez por modelo).

In [ ]:
from tqdm.auto import tqdm

def _cache_path(name, cache_dir): return os.path.join(cache_dir, name)

def save_cache(df: pd.DataFrame, name: str, cache_dir: str = CACHE_DIR) -> str:
    base = _cache_path(name, cache_dir)
    try:
        df.to_parquet(base + ".parquet", index=False); return base + ".parquet"
    except Exception:
        df.to_csv(base + ".csv", index=False); return base + ".csv"

def load_cache(name: str, cache_dir: str = CACHE_DIR) -> Optional[pd.DataFrame]:
    for ext in (".parquet", ".csv"):
        p = _cache_path(name, cache_dir) + ext
        if os.path.exists(p):
            return pd.read_parquet(p) if ext == ".parquet" else pd.read_csv(p)
    return None

def run_inference(name: str, reader: OCRReader, dataset, n: Optional[int] = SAMPLE_N,
                  plain_text: bool = PLAIN_TEXT, cache_dir: str = CACHE_DIR,
                  force: bool = False, sleep_s: float = 0.0) -> pd.DataFrame:
    if not force:
        cached = load_cache(name, cache_dir)
        if cached is not None:
            print(f"[{name}] cache encontrado ({len(cached)} docs) — pulando inferência.")
            return cached
    n = len(dataset) if n is None else min(n, len(dataset))
    rows = []
    for i in tqdm(range(n), desc=name):
        ex = dataset[i]
        res = reader(ex["image_base64"])
        sc = score_pair(ex[GT_COLUMN], res.text, plain_text=plain_text)
        rows.append({"id": ex["id"], "levenshtein_ratio": sc["levenshtein_ratio"],
                     "bleu": sc["bleu"], "score": sc["score"],
                     "textual_degeneration": is_degenerate(res.text, res.hit_token_limit),
                     "latency_s": round(res.latency_s, 3),
                     "num_output_tokens": res.out_tokens, "error": res.error})
        if sleep_s: time.sleep(sleep_s)
    df = pd.DataFrame(rows)
    p = save_cache(df, name, cache_dir)
    n_err = int(df["error"].notna().sum())
    print(f"[{name}] {len(df)} docs, {n_err} erros -> {p}")
    return df

## 5. Agregação por k-fold

Para cada modelo cacheado, particiona os documentos em `KFOLD_K` folds e calcula o `benchmark_score` por fold. Reporta média, desvio e **IC 95%** (t‑Student) entre folds, além do `overall_score` (todos os docs, convenção do paper).

In [ ]:
try:
    from scipy import stats as _stats
    _HAVE_SCIPY = True
except Exception:
    _HAVE_SCIPY = False

def kfold_fold_scores(df: pd.DataFrame, k: int = KFOLD_K, seed: int = SEED) -> np.ndarray:
    d = df[df["error"].isna()] if "error" in df.columns else df
    lev = d["levenshtein_ratio"].to_numpy(); bl = d["bleu"].to_numpy()
    order = np.random.default_rng(seed).permutation(len(d))
    scores = []
    for fold in np.array_split(order, k):
        if len(fold) == 0:
            continue
        scores.append((lev[fold].mean() + bl[fold].mean()) / 2.0)   # score do paper, por fold
    return np.array(scores)

def kfold_summary(df: pd.DataFrame, k: int = KFOLD_K, seed: int = SEED) -> Dict[str, float]:
    fs = kfold_fold_scores(df, k, seed)
    mean = float(fs.mean()); std = float(fs.std(ddof=1)) if len(fs) > 1 else 0.0
    sem = std / math.sqrt(len(fs)) if len(fs) > 1 else 0.0
    if len(fs) > 1:
        tcrit = float(_stats.t.ppf(0.975, len(fs) - 1)) if _HAVE_SCIPY else 1.96
    else:
        tcrit = 0.0
    d = df[df["error"].isna()] if "error" in df.columns else df
    overall = float((d["levenshtein_ratio"].mean() + d["bleu"].mean()) / 2.0)
    deg = float(100.0 * df["textual_degeneration"].mean()) if "textual_degeneration" in df.columns else float("nan")
    return {"n_docs": int(len(d)), "n_error": int(df["error"].notna().sum()) if "error" in df.columns else 0,
            "overall_score": overall, "kfold_mean": mean, "kfold_std": std,
            "ci95": tcrit * sem, "degeneration_pct": deg,
            "latency_s_median": float(df["latency_s"].median()) if "latency_s" in df.columns else float("nan")}

In [ ]:
# Números do paper (Tabela 1 / abstract) para referência. Unlimited-OCR não está no paper;
# o parente de linhagem mais próximo (DeepSeek-OCR) marcou 0.196 — deixado como referência solta.
PAPER_SCORES = {
    "Dharma-OCR-full": 0.925,
    "Dharma-OCR-lite": 0.911,
    "GLM-OCR":         0.710,
    "Unlimited-OCR":   None,      # não avaliado no paper (DeepSeek-OCR ~ 0.196)
}

def compare_models(cache_dir: str = CACHE_DIR, k: int = KFOLD_K, seed: int = SEED) -> pd.DataFrame:
    files = sorted(glob.glob(os.path.join(cache_dir, "*.parquet")) + glob.glob(os.path.join(cache_dir, "*.csv")))
    seen, rows = set(), []
    for f in files:
        name = os.path.splitext(os.path.basename(f))[0]
        if name in seen:
            continue
        seen.add(name)
        df = load_cache(name, cache_dir)
        s = kfold_summary(df, k, seed)
        paper = PAPER_SCORES.get(name)
        s.update({"model": name, "paper_score": paper,
                  "delta_vs_paper": (round(s["overall_score"] - paper, 4) if paper is not None else None)})
        rows.append(s)
    if not rows:
        print("Nenhum cache encontrado em", cache_dir); return pd.DataFrame()
    cols = ["model", "n_docs", "n_error", "overall_score", "kfold_mean", "kfold_std", "ci95",
            "paper_score", "delta_vs_paper", "degeneration_pct", "latency_s_median"]
    out = pd.DataFrame(rows)[cols].sort_values("kfold_mean", ascending=False).reset_index(drop=True)
    return out

### 5.1 Helpers: gráfico por modelo + execução

`run_and_report(name, reader)` roda a inferência (com cache) e já mostra o **gráfico próprio** daquele modelo (score por fold + média/IC95 + linha do paper). `plot_model(name)` re-desenha a partir do cache.

In [ ]:
# --- Gráfico próprio por modelo + driver de execução sequencial ---------------
import matplotlib.pyplot as plt

def plot_model(name: str, k: int = KFOLD_K, seed: int = SEED, cache_dir: str = CACHE_DIR):
    """Gráfico individual do modelo: benchmark_score por fold + média/IC95 e a linha do paper."""
    df = load_cache(name, cache_dir)
    if df is None:
        print("sem cache para", name); return None
    fs = kfold_fold_scores(df, k, seed)
    s  = kfold_summary(df, k, seed)
    paper = PAPER_SCORES.get(name)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(range(1, len(fs) + 1), fs, color="#4C78A8", alpha=0.85, label="score por fold")
    ax.axhline(s["kfold_mean"], color="navy", lw=2, label=f"média = {s['kfold_mean']:.3f}")
    ax.axhspan(s["kfold_mean"] - s["ci95"], s["kfold_mean"] + s["ci95"], color="navy", alpha=0.12,
               label=f"IC95 ±{s['ci95']:.3f}")
    if paper is not None:
        ax.axhline(paper, color="crimson", ls="--", lw=2, label=f"paper = {paper:.3f}")
    ax.set_xticks(range(1, len(fs) + 1)); ax.set_xlabel("fold")
    ax.set_ylabel("DharmaOCR-Benchmark score"); ax.set_ylim(0, 1)
    ax.set_title(f"{name} — score por fold (K={k}, n={s['n_docs']})")
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()
    print(name, "->", {kk: round(v, 4) for kk, v in s.items()
                       if kk in ("n_docs","overall_score","kfold_mean","kfold_std","ci95","degeneration_pct")})
    return s

def run_and_report(name: str, reader: "OCRReader", n: Optional[int] = SAMPLE_N, force: bool = False):
    """Roda a inferência do modelo (com cache) e já mostra o gráfico próprio dele."""
    run_inference(name, reader, ds, n=n, force=force)
    return plot_model(name)

# Leitor 'oráculo' opcional (sanity do pipeline, sem gastar API): descomente p/ testar.
# run_and_report("_oracle", make_oracle_reader(ds), n=40)
print("helpers de gráfico/execução prontos: run_and_report(name, reader), plot_model(name)")

## 6. Rodar os 3 modelos em sequência (cada um com gráfico próprio)

Um modelo por vez — cada célula roda a inferência e mostra o gráfico individual. **Dharma** usa a API hospedada (sem GPU); **GLM-OCR** e **Baidu Unlimited-OCR** sobem via vLLM na GPU (T4 comporta um de cada vez; `start_vllm` → roda → `stop_vllm`).

> Ajuste `SAMPLE_N` (piloto vs. 496). Dharma precisa de `os.environ['DHARMA_API_KEY']`; GLM/Baidu precisam de runtime **GPU** + `%pip install -q vllm`.

In [ ]:
# === 1/3 · Dharma-OCR (API hospedada — NÃO precisa de GPU) ====================
# Cole a chave numa célula à parte:  os.environ["DHARMA_API_KEY"] = "sua-chave"
DHARMA_API_KEY = os.environ.get("DHARMA_API_KEY", "")
if DHARMA_API_KEY:
    run_and_report("Dharma-OCR-full", lambda b: dharma_read(b, "full"))
    run_and_report("Dharma-OCR-lite", lambda b: dharma_read(b, "lite"))
else:
    print("Defina os.environ['DHARMA_API_KEY']='...' e rode esta célula.")

In [ ]:
# === 2/3 · GLM-OCR (vLLM na GPU) =============================================
# Requer runtime GPU + `%pip install -q vllm`. Sobe o modelo, roda, e derruba p/ liberar a VRAM.
try:
    start_vllm("zai-org/GLM-OCR", 8002, "GLM_OCR_URL")
    run_and_report("GLM-OCR", make_openai_ocr_reader(os.environ["GLM_OCR_URL"], "EMPTY", "zai-org/GLM-OCR"))
finally:
    stop_vllm(8002)

In [ ]:
# === 3/3 · Baidu Unlimited-OCR (vLLM na GPU) =================================
try:
    start_vllm("baidu/Unlimited-OCR", 8001, "BAIDU_OCR_URL")
    run_and_report("Unlimited-OCR", make_openai_ocr_reader(os.environ["BAIDU_OCR_URL"], "EMPTY", "baidu/Unlimited-OCR"))
finally:
    stop_vllm(8001)

## 7. Comparativo compilado

Junta todos os modelos que você rodou: **tabela** (score, k-fold mean ± IC95, Δ vs. paper, degeneração), **gráfico combinado** e **teste de significância pareado**.

In [ ]:
from IPython.display import display
# Tabela compilada de TODOS os modelos que você rodou (lê os caches em benchmark_results/)
summary = compare_models()
display(summary)

In [ ]:
import matplotlib.pyplot as plt

if len(summary):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    x = np.arange(len(summary))
    ax.bar(x, summary["kfold_mean"], yerr=summary["ci95"], capsize=6,
           color="#4C78A8", alpha=0.85, label=f"k-fold mean ± IC95 (K={KFOLD_K})")
    for i, row in summary.iterrows():
        if row["paper_score"] is not None and not pd.isna(row["paper_score"]):
            ax.hlines(row["paper_score"], i - 0.4, i + 0.4, colors="crimson",
                      linestyles="--", linewidth=2)
    ax.plot([], [], color="crimson", linestyle="--", label="paper score")
    ax.set_xticks(x); ax.set_xticklabels(summary["model"], rotation=20, ha="right")
    ax.set_ylabel("DharmaOCR-Benchmark score"); ax.set_ylim(0, 1)
    ax.set_title("OCRs na figura de mérito do paper — média por k-fold ± IC95 vs. paper")
    ax.legend(); ax.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()
else:
    print("Rode a Seção 4 (inferência) para popular o cache antes de plotar.")

### 7.1 Significância pareada

Score por documento pareado por `id` entre dois modelos — Wilcoxon + t pareado.

In [ ]:
def paired_significance(name_a: str, name_b: str, cache_dir: str = CACHE_DIR) -> Optional[Dict[str, Any]]:
    da, db = load_cache(name_a, cache_dir), load_cache(name_b, cache_dir)
    if da is None or db is None:
        print("Faltam caches para", name_a, "ou", name_b); return None
    m = da[["id", "score"]].merge(db[["id", "score"]], on="id", suffixes=(f"_{name_a}", f"_{name_b}"))
    a = m[f"score_{name_a}"].to_numpy(); b = m[f"score_{name_b}"].to_numpy()
    out = {"n_pairs": int(len(m)), "mean_diff": float((a - b).mean()),
           "median_diff": float(np.median(a - b))}
    if _HAVE_SCIPY and len(m) > 1:
        try:
            out["wilcoxon_p"] = float(_stats.wilcoxon(a, b, zero_method="wilcox").pvalue)
        except ValueError as e:
            out["wilcoxon_p"] = f"n/a ({e})"
        out["ttest_p"] = float(_stats.ttest_rel(a, b).pvalue)
    return out

# Exemplo (ajuste aos modelos que você rodou):
_names = list((summary["model"] if len(summary) else pd.Series([], dtype=str)))
if len(_names) >= 2:
    print(f"{_names[0]} vs {_names[1]}:",
          json.dumps(paired_significance(_names[0], _names[1]), ensure_ascii=False, indent=2))
else:
    print("Rode ao menos 2 modelos para o teste pareado.")

## 8. Próximos passos

- Configure as envs (`DHARMA_API_KEY`; `BAIDU_OCR_URL`; `GLM_API_KEY`) e rode a Seção 4 — comece com `SAMPLE_N = 40` (piloto) para validar contrato/latência antes dos 496.
- Suba `SAMPLE_N = None` para o benchmark completo; o k-fold (Seções 5–7) dá média ± IC95 por modelo e o `delta_vs_paper`.
- Para robustez extra, repita o k-fold com várias sementes (`SEED`) e reporte média dos means — ou troque para bootstrap dos scores por-doc.
- Quebra por subconjunto (ESTER‑Pt/Legal/BRESSAY) exige os rótulos por linha (não vêm no release) — pedir à Dharma se for necessário.
